# AH | 01. Investigación exploratoria de Integración y Reporting

**Objetivo:** entender qué contiene cada archivo, qué problemas de calidad presenta y qué reglas deben utilizarse en el ETL.

Cada fila original representa **una visita**. Las aperturas y clics son contadores separados.

El análisis solamente lee los archivos de `input` y el catálogo de permisos. **No escribe salidas ni conecta al SFTP**. Se usan tablas, gráficos y celdas pequeñas para explicar cada decisión.

## 01. Ruta y bibliotecas
Una sola ruta configurable. Los archivos de entrada se encuentran en la carpeta del proyecto.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd().resolve()
if BASE.name.lower() == "notebooks":
    BASE = BASE.parent
INPUT = BASE / "input"
CATALOGOS = BASE / "Catalogos"

assert INPUT.is_dir() and CATALOGOS.is_dir(), "Inicia Jupyter desde la raiz del proyecto o desde notebooks/."

## 02. ¿Cuántos archivos recibimos?
El patrón `report_*.txt` permite incorporar futuros reportes sin editar el análisis.

In [ ]:
archivos = sorted(INPUT.glob("report_*.txt"), key=lambda a: int(a.stem.split("_")[1]))
assert archivos, "No hay archivos report_*.txt"
pd.DataFrame({"Archivo": [a.name for a in archivos]})

## 03. Leer los TXT originales
A pesar de la extensión TXT, los archivos usan **coma** como delimitador. Leemos todo como texto para investigar los valores sin convertirlos por accidente.

In [ ]:
reportes = []
for archivo in archivos:
    reporte = pd.read_csv(archivo, sep=",", dtype=str, keep_default_na=False,
                          encoding="utf-8-sig")
    reportes.append(reporte)

pd.DataFrame({"Archivo": [a.name for a in archivos],
              "Filas": [len(r) for r in reportes],
              "Columnas": [r.shape[1] for r in reportes]})

## 04. ¿Qué encabezados coinciden?
Comparamos **nombres de columnas por posición**, no los valores de sus registros. El segundo encabezado es variable (`jk`, `jyv`, `fgh`).

In [ ]:
encabezados = pd.DataFrame({a.name: r.columns for a, r in zip(archivos, reportes)})
encabezados["Coinciden"] = encabezados.nunique(axis=1) == 1
encabezados.index = encabezados.index + 1
encabezados

**Decisión:** conservar el segundo campo bajo el nombre común `CampoExtra` en el ETL. Ningún otro encabezado debería variar sin revisión.

## 05. Filas repetidas dentro de cada archivo
Un registro idéntico es una alerta para investigar; **no eliminamos visitas automáticamente**.

In [ ]:
pd.DataFrame({"Archivo": [a.name for a in archivos],
              "Filas": [len(r) for r in reportes],
              "Duplicados completos": [r.duplicated().sum() for r in reportes]})

## 06. Valores ausentes: vacío, guion y `unknown`
En estos TXT, una cadena vacía y `-` son distintas de un valor conocido. `unknown` también merece control específico.

In [ ]:
calidad = []
for archivo, df in zip(archivos, reportes):
    for col in df.columns:
        serie = df[col].str.strip()
        calidad.append([archivo.name, col, serie.eq("").sum(),
                        serie.eq("-").sum(), serie.str.lower().eq("unknown").sum()])
calidad = pd.DataFrame(calidad, columns=["Archivo", "Campo", "Vacíos", "Guiones", "Unknown"])
calidad.query("Vacíos > 0 or Guiones > 0 or Unknown > 0")

## 07. Usuarios y repetición de correos
Diferenciamos **visitas (filas)** de **usuarios (correos normalizados)**. Repetir un correo no necesariamente significa un duplicado.

In [ ]:
correos = []
for archivo, df in zip(archivos, reportes):
    normalizados = df["email"].str.strip().str.lower()
    correos.append([archivo.name, len(df), normalizados.nunique(),
                    normalizados.duplicated().sum()])
pd.DataFrame(correos, columns=["Archivo", "Visitas", "Correos únicos", "Repeticiones"] )

## 08. Unir temporalmente para investigar
`muestra` existe **solo en memoria**. No es una exportación ni el ETL. Unificamos el segundo encabezado por su posición únicamente para estudiar las 2,001 filas juntas.

In [ ]:
partes = []
for archivo, df in zip(archivos, reportes):
    copia = df.copy()
    copia.columns = list(copia.columns[:1]) + ["CampoExtra"] + list(copia.columns[2:])
    copia["ArchivoOrigen"] = archivo.name
    partes.append(copia)
muestra = pd.concat(partes, ignore_index=True)
muestra[["ArchivoOrigen", "email", "Opens", "Clicks"]].head()

## 09. Indicadores originales
No confundimos `Opens` con visitas: la granularidad definida para el caso es **una visita por fila recibida**.

In [ ]:
for campo in ["Opens", "Opens virales", "Clicks", "Clicks virales"]:
    muestra[campo] = pd.to_numeric(muestra[campo], errors="coerce").fillna(0)

pd.DataFrame({"Indicador": ["Visitas", "Usuarios únicos", "Aperturas", "Aperturas virales", "Clics", "Clics virales"],
              "Resultado": [len(muestra), muestra["email"].str.strip().str.lower().nunique(),
                            muestra["Opens"].sum(), muestra["Opens virales"].sum(),
                            muestra["Clicks"].sum(), muestra["Clicks virales"].sum()]})

## 10. Primer gráfico: tamaño de los archivos
Ayuda a explicar la procedencia del volumen, no su evolución temporal.

In [ ]:
volumen = muestra.groupby("ArchivoOrigen").size().sort_index()
ax = volumen.plot.bar(figsize=(8, 3.5), rot=0)
ax.set(title="Registros por archivo", xlabel="Archivo TXT", ylabel="Visitas")
for i, n in enumerate(volumen): ax.text(i, n + 15, str(n), ha="center")
plt.tight_layout()
plt.show()

## 11. ¿De dónde podemos obtener la fecha?
La prioridad de fecha es **apertura → envío → procesamiento**. Aquí solo calculamos cuántos registros necesitarán cada respaldo.

In [ ]:
apertura = pd.to_datetime(muestra["Fecha open"], dayfirst=True, errors="coerce")
envio = pd.to_datetime(muestra["Fecha envio"], dayfirst=True, errors="coerce")

muestra["FechaEstudio"] = apertura.fillna(envio)
muestra["OrigenEstudio"] = "PROCESAMIENTO"
muestra.loc[envio.notna(), "OrigenEstudio"] = "ENVIO"
muestra.loc[apertura.notna(), "OrigenEstudio"] = "OPEN"
muestra["OrigenEstudio"].value_counts().rename_axis("Origen").reset_index(name="Visitas")

## 12. Segundo gráfico: calidad de la fecha analítica
La fecha de envío se utiliza como respaldo cuando falta la de apertura; no indica que se haya producido una apertura.

In [ ]:
origen = muestra["OrigenEstudio"].value_counts()
ax = origen.plot.bar(figsize=(7, 3.5), rot=0)
ax.set(title="Origen de la fecha de visita", ylabel="Registros", xlabel="Regla aplicada")
for i, n in enumerate(origen): ax.text(i, n + 20, str(n), ha="center")
plt.tight_layout()
plt.show()

## 13. ¿Qué días abarca la muestra?
Los registros corresponden a febrero de 2013. Las tendencias se interpretan dentro del período disponible.

In [ ]:
visitas_dia = muestra.groupby(muestra["FechaEstudio"].dt.date).size()
visitas_dia.rename("Visitas").reset_index().rename(columns={"FechaEstudio":"Día"})

## 14. Tercer gráfico: evolución diaria
Aquí sí utilizamos la fecha calculada en memoria para observar cómo se distribuyeron las visitas.

In [ ]:
ax = visitas_dia.plot(figsize=(9, 3.5), marker="o")
ax.set(title="Visitas por día · muestra histórica", xlabel="Día", ylabel="Visitas")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 15. Fechas cronológicamente anómalas
Podemos encontrar aperturas anteriores al envío. No borramos esos registros: documentamos el problema y lo explicamos como limitación de calidad.

In [ ]:
pd.DataFrame({"Control": ["OPEN antes de ENVIO", "Sin OPEN y sin ENVIO", "Fechas analíticas válidas"],
              "Registros": [(apertura.notna() & envio.notna() & (apertura < envio)).sum(),
                             (apertura.isna() & envio.isna()).sum(),
                             muestra["FechaEstudio"].notna().sum()]})

## 16. Navegadores y plataformas: información conocida y desconocida
Los navegadores son valores originales; las plataformas podrán aparecer como listas separadas por comas.

In [ ]:
pd.DataFrame({"Control": ["Plataforma guion", "Plataforma unknown", "Plataforma con coma", "Navegador guion"],
              "Registros": [muestra["Plataformas"].eq("-").sum(),
                             muestra["Plataformas"].str.lower().eq("unknown").sum(),
                             muestra["Plataformas"].str.contains(",", regex=False).sum(),
                             muestra["Navegadores"].eq("-").sum()]})

## 17. Explorar las plataformas individuales
Esta separación sirve para **investigar** las categorías. El ETL construirá después la tabla auxiliar `VisitaPlataforma`; la tabla principal seguirá teniendo una fila por visita.

In [ ]:
plataformas = []
for celda in muestra["Plataformas"]:
    for token in celda.split(","):
        token = token.strip()
        if token and token != "-" and token.lower() != "unknown":
            plataformas.append(token)
plataformas_tabla = pd.Series(plataformas).value_counts().rename("Asociaciones")
plataformas_tabla.rename_axis("Plataforma").reset_index()

## 18. Cuarto gráfico: plataformas identificadas
La suma de las barras representa **asociaciones** y puede superar el número de visitas identificadas, porque hay visitas con varias plataformas.

In [ ]:
ax = plataformas_tabla.sort_values().plot.barh(figsize=(8, 4))
ax.set(title="Asociaciones por plataforma", xlabel="Asociaciones", ylabel="Plataforma")
plt.tight_layout()
plt.show()

## 19. ¿Qué tan diferentes son las aperturas y los clics?
Ambas son métricas de actividad; se conciliarán con el dashboard final.

In [ ]:
actividad = muestra.groupby("ArchivoOrigen")[["Opens", "Clicks"]].sum()
actividad.rename(columns={"Opens": "Aperturas", "Clicks": "Clics"})

## 20. Quinto gráfico: actividad por origen
Gráfico comparativo; no interpretarlo como una tasa de conversión ni como métricas de usuarios únicos.

In [ ]:
ax = actividad.plot.bar(figsize=(8, 3.5), rot=0)
ax.set(title="Aperturas y clics por archivo", xlabel="Archivo", ylabel="Eventos")
plt.tight_layout()
plt.show()

## 21. Catálogo de permisos RLS
Aquí **sí corresponde un catálogo**: `usuarios.txt` contiene una fila por asociación entre usuario y plataforma permitida. No se inventa un maestro adicional.

In [ ]:
usuarios = pd.read_csv(CATALOGOS / "usuarios.txt", dtype=str)
usuarios.groupby(["Usuario", "Rol"])["Plataforma"].agg(list).reset_index()

## 22. Controles finales del análisis
Estos valores son la referencia **de los tres TXT originales**. Cuando entren reportes nuevos, los totales cambiarán legítimamente.

In [ ]:
control = pd.DataFrame({
    "Control": ["Archivos", "Filas", "Encabezados coincidentes", "Usuarios únicos",
                "Aperturas", "Clics", "Asociaciones de plataforma"],
    "Observado": [len(archivos), len(muestra), int(encabezados["Coinciden"].sum()),
                  muestra["email"].str.strip().str.lower().nunique(),
                  int(muestra["Opens"].sum()), int(muestra["Clicks"].sum()), len(plataformas)]})
control

## 23. Clics promedio por navegador
El indicador del tablero es `Clicks / Visitas`. Para evitar conclusiones con muestras de uno o dos registros, el gráfico se limita a navegadores con **cinco visitas o más**, sin borrar esos registros de la fuente.

In [ ]:
por_navegador = muestra.groupby("Navegadores").agg(Visitas=("email", "size"), Clics=("Clicks", "sum"))
por_navegador = por_navegador.loc[por_navegador["Visitas"] >= 5].copy()
por_navegador["ClicsPorVisita"] = por_navegador["Clics"] / por_navegador["Visitas"]
por_navegador.sort_values("ClicsPorVisita", ascending=False).head(10)

## 24. Sexto gráfico: actividad por navegador
La longitud representa clics **promedio por visita**, no visitas totales. Los promedios bajos son normales en esta muestra.

In [ ]:
top_navegadores = por_navegador.nlargest(10, "Visitas")
ax = top_navegadores.sort_values("ClicsPorVisita")["ClicsPorVisita"].plot.barh(figsize=(9, 4))
ax.set(title="Clics por visita según navegador (mínimo 5 visitas)", xlabel="Clics por visita")
plt.tight_layout()
plt.show()